## 1. Configuration

In [1]:
import os
from functools import reduce

import numpy as np
import pandas as pd
import tensorflow as tf
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Masking, GRU, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.sequence import pad_sequences

import feature_extraction as fe   # module de l'équipe (ingest_all)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

MANIFEST_PATH = "/kaggle/usr/lib/notebooks/rouatabarki/sequence_generation/manifest.csv"
WORK = "/kaggle/working"

STREAMS   = ["ata_read", "ata_write", "mem_exec", "mem_readwrite", "mem_read", "mem_write"]
META_COLS = ["rel_dir", "family", "top", "cpu", "ram", "hw", "label"]
FRACTIONS = [0.10, 0.25, 0.50, 0.75]
PAD_VALUE = -999.0

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


## 2. Manifest et séparation train / val / test



In [2]:
manifest = pd.read_csv(MANIFEST_PATH)

manifests = {name: manifest[manifest["split"] == name].copy()
             for name in ["train", "val", "test"]}

for name, df in manifests.items():
    print(f"{name:5s}", df.shape, "| ransomware:", int(df["label"].sum()), "| benign:", int((df["label"] == 0).sum()))

train (770, 16) | ransomware: 350 | benign: 420
val   (220, 16) | ransomware: 100 | benign: 120
test  (330, 16) | ransomware: 150 | benign: 180


## 3. Session Spark

In [3]:
spark = (
    SparkSession.builder
    .appName("RanSMAP_Feature_Extraction")
    .master("local[*]")
    .config("spark.driver.memory", "20g")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 08:51:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.0.4


## 4. Extraction des features (CSV bruts → fenêtres d'1 seconde)



In [4]:
RUN_EXTRACTION = True

if RUN_EXTRACTION:
    reports = {}
    for name, df in manifests.items():
        reports[name] = fe.ingest_all(
            exec_df=df,
            spark=spark,
            out_root=f"{WORK}/manifest_{name}",
        )


Processing stream: ata_read
Total executions: 770
Available files: 769
Missing files: 1
Missing: /kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset/original/celeron-gen6/ddr4-2133-16g/LockBit/LockBit-20221006_16-38-01/ata_read.csv


Completed ata_read
Windows: 126485
Output: /kaggle/working/manifest_train/ata_read
Time: 761.72 seconds

Processing stream: ata_write
Total executions: 770
Available files: 770
Missing files: 0


Completed ata_write
Windows: 136040
Output: /kaggle/working/manifest_train/ata_write
Time: 1464.6 seconds

Processing stream: mem_exec
Total executions: 770
Available files: 770
Missing files: 0


Completed mem_exec
Windows: 9065
Output: /kaggle/working/manifest_train/mem_exec
Time: 31.82 seconds

Processing stream: mem_readwrite
Total executions: 770
Available files: 770
Missing files: 0


Completed mem_readwrite
Windows: 34958
Output: /kaggle/working/manifest_train/mem_readwrite
Time: 29.75 seconds

Processing stream: mem_read
Total executions: 770
Available files: 770
Missing files: 0


Completed mem_read
Windows: 151945
Output: /kaggle/working/manifest_train/mem_read
Time: 810.37 seconds

Processing stream: mem_write
Total executions: 770
Available files: 770
Missing files: 0


... (9 outputs left out)

Completed mem_exec
Windows: 4669
Output: /kaggle/working/manifest_test/mem_exec
Time: 14.12 seconds

Processing stream: mem_readwrite
Total executions: 330
Available files: 330
Missing files: 0


Completed mem_readwrite
Windows: 18070
Output: /kaggle/working/manifest_test/mem_readwrite
Time: 14.01 seconds

Processing stream: mem_read
Total executions: 330
Available files: 330
Missing files: 0


Completed mem_read
Windows: 64618
Output: /kaggle/working/manifest_test/mem_read
Time: 314.44 seconds

Processing stream: mem_write
Total executions: 330
Available files: 330
Missing files: 0


Completed mem_write
Windows: 64669
Output: /kaggle/working/manifest_test/mem_write
Time: 339.0 seconds

All streams processed.


## 5. Fusion des 6 flux + métadonnées + `step`



In [5]:
def build_split(name, manifest_df):
    """Fusionne les 6 flux, ajoute les métadonnées, remplit les NULL par 0,
    calcule `step` (seconde relative au début de l'exécution) et écrit en Parquet."""
    out_root = f"{WORK}/manifest_{name}"

    dfs = []
    for s in STREAMS:
        d = (spark.read.parquet(f"{out_root}/{s}")
             .filter(F.col("win").isNotNull())
             .drop(f"{s}_rawcnt"))
        dfs.append(d)

    merged = reduce(lambda a, b: a.join(b, ["rel_dir", "win"], "full_outer"), dfs)

    meta = spark.createDataFrame(manifest_df[META_COLS].copy())
    win = merged.join(F.broadcast(meta), on="rel_dir", how="inner")

    # NULL = aucun événement dans ce flux pendant cette seconde -> 0
    num_cols = [c for c in win.columns if c not in META_COLS + ["win"]]
    win = win.fillna(0, subset=num_cols)

    win = (win.withColumn("step", F.col("win") - F.min("win").over(Window.partitionBy("rel_dir")))
              .drop("win"))

    path = f"{WORK}/merged_{name}"
    win.write.mode("overwrite").parquet(path)
    return path, len(num_cols)


paths = {}
for name, df in manifests.items():
    paths[name], n_feat = build_split(name, df)
    print(f"{name:5s} -> {paths[name]}  ({n_feat} features)")

train -> /kaggle/working/merged_train  (24 features)
val   -> /kaggle/working/merged_val  (24 features)
test  -> /kaggle/working/merged_test  (24 features)


## 6. Contrôle qualité : trous dans `step`



In [6]:
for name, path in paths.items():
    d = spark.read.parquet(path)
    per = d.groupBy("rel_dir").agg(F.count("*").alias("n"), F.max("step").alias("smax"))
    n_exec = per.count()
    n_gaps = per.filter(F.col("smax") + 1 != F.col("n")).count()
    print(f"{name:5s} exécutions: {n_exec:5d} | lignes: {d.count():>9,} | exécutions avec trous: {n_gaps}")

train exécutions:   770 | lignes:   152,064 | exécutions avec trous: 18
val   exécutions:   220 | lignes:    43,444 | exécutions avec trous: 7
test  exécutions:   330 | lignes:    64,670 | exécutions avec trous: 10


## 7. Spark → pandas

In [7]:
train_pd = spark.read.parquet(paths["train"]).toPandas()
val_pd   = spark.read.parquet(paths["val"]).toPandas()
test_pd  = spark.read.parquet(paths["test"]).toPandas()

feature_cols = [c for c in train_pd.columns if c not in META_COLS + ["step"]]
print("Nombre de features:", len(feature_cols))

for name, df in [("Train", train_pd), ("Val", val_pd), ("Test", test_pd)]:
    lengths = df.groupby("rel_dir").size()
    print(f"{name:5s} labels: {df.groupby('rel_dir')['label'].first().value_counts().to_dict()} "
          f"| longueur min/moy/max: {lengths.min()}/{lengths.mean():.1f}/{lengths.max()}")

Nombre de features: 24
Train labels: {0: 420, 1: 350} | longueur min/moy/max: 43/197.5/280
Val   labels: {0: 120, 1: 100} | longueur min/moy/max: 9/197.5/264
Test  labels: {0: 180, 1: 150} | longueur min/moy/max: 164/196.0/322


## 8. Séquences (une par exécution)

In [8]:
def create_sequences(df, feature_cols, fill_gaps=True):
    """Retourne (liste de matrices [n_secondes x n_features], labels, ids).
    fill_gaps=True : les secondes sans événement sont réinsérées avec des 0."""
    sequences, labels, ids = [], [], []
    for exec_id, group in df.groupby("rel_dir"):
        g = group.sort_values("step").set_index("step")[feature_cols]
        if fill_gaps:
            g = g.reindex(range(int(g.index.max()) + 1), fill_value=0)
        sequences.append(g.to_numpy(dtype=np.float32))
        labels.append(int(group["label"].iloc[0]))
        ids.append(exec_id)
    return sequences, np.array(labels), ids


X_train, y_train, train_ids = create_sequences(train_pd, feature_cols)
X_val,   y_val,   val_ids   = create_sequences(val_pd,   feature_cols)
X_test,  y_test,  test_ids  = create_sequences(test_pd,  feature_cols)

print("Séquences  train/val/test:", len(X_train), len(X_val), len(X_test))

# Aucune exécution ne doit apparaître dans deux jeux
assert not (set(train_ids) & set(val_ids))
assert not (set(train_ids) & set(test_ids))
assert not (set(val_ids) & set(test_ids))
print("Pas de chevauchement entre les jeux")

Séquences  train/val/test: 770 220 330
Pas de chevauchement entre les jeux


## 9. Normalisation (ajustée sur le train uniquement)

In [9]:
scaler = StandardScaler().fit(np.concatenate(X_train, axis=0))

X_train_s = [scaler.transform(s) for s in X_train]
X_val_s   = [scaler.transform(s) for s in X_val]
X_test_s  = [scaler.transform(s) for s in X_test]

## 10. Préfixes (début de l'exécution) + padding



In [10]:
def make_prefixes(sequences, fraction):
    return [seq[:max(1, int(len(seq) * fraction))] for seq in sequences]


padded_data = {}
for fraction in FRACTIONS:
    train_p = make_prefixes(X_train_s, fraction)
    val_p   = make_prefixes(X_val_s,   fraction)
    test_p  = make_prefixes(X_test_s,  fraction)

    max_len = max(len(s) for s in train_p + val_p + test_p)
    pad = lambda seqs: pad_sequences(seqs, maxlen=max_len, padding="post",
                                     dtype="float32", value=PAD_VALUE)

    padded_data[fraction] = {"train": pad(train_p), "val": pad(val_p), "test": pad(test_p)}
    print(f"{int(fraction*100):3d}% -> train {padded_data[fraction]['train'].shape} | "
          f"val {padded_data[fraction]['val'].shape} | test {padded_data[fraction]['test'].shape}")

 10% -> train (770, 32, 24) | val (220, 32, 24) | test (330, 32, 24)
 25% -> train (770, 80, 24) | val (220, 80, 24) | test (330, 80, 24)
 50% -> train (770, 161, 24) | val (220, 161, 24) | test (330, 161, 24)
 75% -> train (770, 241, 24) | val (220, 241, 24) | test (330, 241, 24)


## 11. Modèles

In [11]:
def build_model(cell, input_shape, units=64):
    model = Sequential([
        Input(shape=input_shape),
        Masking(mask_value=PAD_VALUE),
        cell(units),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


MODELS = {"GRU": GRU, "LSTM": LSTM}


def train_and_evaluate(model_name, fraction, epochs=30, batch_size=32):
    data = padded_data[fraction]
    model = build_model(MODELS[model_name], data["train"].shape[1:])

    early_stopping = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    history = model.fit(
        data["train"], y_train,
        validation_data=(data["val"], y_val),
        epochs=epochs, batch_size=batch_size,
        callbacks=[early_stopping], verbose=0,
    )

    proba = model.predict(data["test"], verbose=0)
    pred = (proba >= 0.5).astype(int).ravel()

    metrics = {
        "Model": model_name,
        "Prefix": f"{int(fraction * 100)}%",
        "Epochs": len(history.history["loss"]),
        "Accuracy":  accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall":    recall_score(y_test, pred, zero_division=0),
        "F1-score":  f1_score(y_test, pred, zero_division=0),
    }
    model.save(f"{WORK}/{model_name.lower()}_{int(fraction * 100)}_model.keras")
    return metrics, confusion_matrix(y_test, pred), history

## 12. Entraînement : 2 modèles × 4 préfixes

In [12]:
results, confusions, histories = [], {}, {}

for model_name in MODELS:
    for fraction in FRACTIONS:
        metrics, cm, history = train_and_evaluate(model_name, fraction)
        results.append(metrics)
        confusions[(model_name, fraction)] = cm
        histories[(model_name, fraction)] = history
        print(f"{model_name:4s} {metrics['Prefix']:>4s} | "
              f"acc={metrics['Accuracy']:.3f} prec={metrics['Precision']:.3f} "
              f"recall={metrics['Recall']:.3f} f1={metrics['F1-score']:.3f} "
              f"(epochs={metrics['Epochs']})")

results_df = pd.DataFrame(results)
results_df.to_csv(f"{WORK}/results_gru_lstm.csv", index=False)
results_df

GRU   10% | acc=0.955 prec=0.947 recall=0.953 f1=0.950 (epochs=19)
GRU   25% | acc=1.000 prec=1.000 recall=1.000 f1=1.000 (epochs=16)
GRU   50% | acc=0.985 prec=1.000 recall=0.967 f1=0.983 (epochs=23)
GRU   75% | acc=1.000 prec=1.000 recall=1.000 f1=1.000 (epochs=30)
LSTM  10% | acc=0.939 prec=0.928 recall=0.940 f1=0.934 (epochs=11)
LSTM  25% | acc=1.000 prec=1.000 recall=1.000 f1=1.000 (epochs=13)
LSTM  50% | acc=0.994 prec=0.993 recall=0.993 f1=0.993 (epochs=14)
LSTM  75% | acc=1.000 prec=1.000 recall=1.000 f1=1.000 (epochs=30)


,Model,Prefix,Epochs,Accuracy,Precision,Recall,F1-score
0,GRU,10%,19,0.954545,0.947020,0.953333,0.950166
1,GRU,25%,16,1.000000,1.000000,1.000000,1.000000
2,GRU,50%,23,0.984848,1.000000,0.966667,0.983051
3,GRU,75%,30,1.000000,1.000000,1.000000,1.000000
4,LSTM,10%,11,0.939394,0.927632,0.940000,0.933775
5,LSTM,25%,13,1.000000,1.000000,1.000000,1.000000
6,LSTM,50%,14,0.993939,0.993333,0.993333,0.993333
7,LSTM,75%,30,1.000000,1.000000,1.000000,1.000000


## 13. Comparaison

In [13]:
results_df.pivot(index="Prefix", columns="Model", values=["Recall", "F1-score"]).round(3)

Recall        F1-score       
Model     GRU   LSTM      GRU   LSTM
Prefix                              
10%     0.953  0.940    0.950  0.934
25%     1.000  1.000    1.000  1.000
50%     0.967  0.993    0.983  0.993
75%     1.000  1.000    1.000  1.000

In [14]:
for (model_name, fraction), cm in confusions.items():
    print(f"{model_name} {int(fraction*100)}%  [[TN FP] [FN TP]]:\n{cm}\n")

GRU 10%  [[TN FP] [FN TP]]:
[[172   8]
 [  7 143]]

GRU 25%  [[TN FP] [FN TP]]:
[[180   0]
 [  0 150]]

GRU 50%  [[TN FP] [FN TP]]:
[[180   0]
 [  5 145]]

GRU 75%  [[TN FP] [FN TP]]:
[[180   0]
 [  0 150]]

LSTM 10%  [[TN FP] [FN TP]]:
[[169  11]
 [  9 141]]

LSTM 25%  [[TN FP] [FN TP]]:
[[180   0]
 [  0 150]]

LSTM 50%  [[TN FP] [FN TP]]:
[[179   1]
 [  1 149]]

LSTM 75%  [[TN FP] [FN TP]]:
[[180   0]
 [  0 150]]

